In [1]:
!pip3 install -q transformers torch accelerate sentencepiece


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [5]:
import pandas as pd 
train_df = pd.read_csv('data/6/train.csv')
train_df

,sentiment,text,data_id
0,positive,Best sock ever made. Consistent quality...I hi...,6
1,positive,It always makes the water taste so good,6
2,positive,Looks good but did not need. Put into stock,6
3,negative,Unit worked perfectly for approximately 6 days...,6
4,positive,They fit perfectly on my ancient General Elect...,6
...,...,...,...
4909,negative,Bulky and slow. May change opinion in the summ...,6
4910,positive,The quality was good,6
4911,positive,"It's a filter, it was fine.",6
4912,positive,Love that it is so easy to use.,6


In [ ]:
# =========================
# 1-sample LLM classification pipeline
# =========================

import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------
# 1. Config
# -------------------------
MODEL_NAME = "google/flan-t5-small"   # example open-source instruction model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataset labels
VALID_LABELS = train_df['sentiment'].unique().tolist()

# One sample only
sample_text = train_df['text'][0]
true_label = train_df['sentiment'][0]

# -------------------------
# 2. Prompt template
# -------------------------
def build_prompt(text: str) -> str:
    return f"""
        You are a text classification system.

        Task:
        Classify the given text into exactly one label from:
        {", ".join(VALID_LABELS)}

        Rules:
        - Return only one label
        - Do not explain
        - Do not write anything else

        Text:
        {text}

        Label:
        """.strip()

# -------------------------
# 3. Post-processing
# -------------------------
def normalize_label(raw_output: str, valid_labels: list[str]) -> str:
    text = raw_output.strip().lower()

    # exact match first
    if text in valid_labels:
        return text

    # first valid label appearing in output
    for label in valid_labels:
        if re.search(rf"\b{re.escape(label.lower())}\b", text):
            return label

    return "UNKNOWN"

# -------------------------
# 4. Load tokenizer/model
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

# For some causal models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# 5. Build prompt
# -------------------------
prompt = build_prompt(sample_text)
print("PROMPT:\n", prompt)

# -------------------------
# 6. Tokenize
# -------------------------
inputs = tokenizer( # Text -> Token IDs
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(DEVICE)

# -------------------------
# 7. Generate
# -------------------------
with torch.no_grad(): # no need to track gradients for inference (saves memory and computations and speeds up)
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,      # deterministic for classification
        temperature=None,
        pad_token_id=tokenizer.eos_token_id
    )

# -------------------------
# 8. Decode only generated part
# -------------------------
raw_prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip() # Token IDs -> Text

# -------------------------
# 9. Post-process to final label
# -------------------------
pred_label = normalize_label(raw_prediction, VALID_LABELS)

# -------------------------
# 10. Compare
# -------------------------
print("\nRAW MODEL OUTPUT:", raw_prediction)
print("PREDICTED LABEL :", pred_label)
print("TRUE LABEL      :", true_label)
print("CORRECT?        :", pred_label == true_label)

PROMPT:
 You are a text classification system.

        Task:
        Classify the given text into exactly one label from:
        positive, negative

        Rules:
        - Return only one label
        - Do not explain
        - Do not write anything else

        Text:
        Best sock ever made. Consistent quality...I highly recommend Gold Toe  socks.

        Label:

RAW MODEL OUTPUT: positive
PREDICTED LABEL : positive
TRUE LABEL      : positive
CORRECT?        : True
